# Actividad Semana 4 - Fundamentos para IA
## Librerías de cálculo numérico y visualización

**Estudiante:** Kevin Andres Granados Daza  
**Dataset:** Netflix TV Shows and Movies

En este taller se trabajan los archivos `titles.csv` y `credits.csv` del dataset de Netflix. La idea es aplicar Pandas, NumPy, SciPy, Matplotlib y Seaborn para explorar, organizar y visualizar los datos.

## 1. Frameworks y librerías de IA

Una librería es un conjunto de funciones que se importa para hacer una tarea puntual. Por ejemplo, NumPy ayuda con cálculos numéricos y Pandas facilita el manejo de tablas. Un framework es más amplio: trae una estructura de trabajo y herramientas conectadas para construir una solución completa.

- **Scikit-learn:** se utiliza para clasificación, regresión, clustering y evaluación de modelos clásicos de machine learning.
- **TensorFlow:** permite construir y entrenar redes neuronales; se aplica en visión, predicción y texto.
- **PyTorch:** también se usa para deep learning, visión por computador y procesamiento de lenguaje natural.
- **Hugging Face:** ofrece modelos ya entrenados para tareas de texto, audio, imágenes y generación de contenido.

En pocas palabras, las librerías ayudan a resolver partes específicas y los frameworks sirven para organizar proyectos de IA más completos.

## 2. Carga del dataset

Los datos se cargan directamente desde el repositorio de GitHub. Así el notebook no depende de una ruta local y se puede ejecutar en otro computador mientras tenga acceso a internet.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

base_url = (
    'https://raw.githubusercontent.com/cesarabs54/'
    'NRC-70446-Fundamentos-para-IA/'
    'e62e6d966a9704092a7121d149d44ec4edb2d6e4/'
    'Semana_4/Semana_4_Actividad/'
)

titles = pd.read_csv(base_url + 'titles.csv')
credits = pd.read_csv(base_url + 'credits.csv')

print('Los archivos se cargaron correctamente.')

## Exploración inicial

Primero reviso las primeras filas, la cantidad de registros y los tipos de datos de cada archivo.

In [ ]:
display(titles.head())
print('Dimensión de titles:', titles.shape)
titles.info()

In [ ]:
display(credits.head())
print('Dimensión de credits:', credits.shape)
credits.info()

## 3. Limpieza y selección de variables

Voy a trabajar principalmente con `titles.csv`, porque allí están las variables numéricas necesarias para las estadísticas y las gráficas. Las variables escogidas son `imdb_score` e `imdb_votes` como numéricas, y `type` como categórica.

Se convierten a formato numérico las columnas necesarias. Para los cálculos que dependen de puntuación y votos, se excluyen únicamente las filas que tengan nulos en esas dos columnas. No se imputan las calificaciones porque inventar un valor podría cambiar los resultados del análisis.

In [ ]:
titles.columns = titles.columns.str.strip().str.lower()
credits.columns = credits.columns.str.strip().str.lower()

columnas_numericas = [
    'release_year', 'runtime', 'imdb_score',
    'imdb_votes', 'tmdb_score', 'tmdb_popularity'
]

for columna in columnas_numericas:
    if columna in titles.columns:
        titles[columna] = pd.to_numeric(titles[columna], errors='coerce')

print('Valores nulos en variables seleccionadas:')
display(titles[['imdb_score', 'imdb_votes', 'type']].isna().sum().to_frame('cantidad'))

datos = titles.dropna(subset=['imdb_score', 'imdb_votes']).copy()
print('Filas originales:', len(titles))
print('Filas utilizadas en el análisis:', len(datos))

display(datos[['title', 'type', 'imdb_score', 'imdb_votes']].head())

## 4. Operaciones con NumPy

En esta parte convierto columnas a arreglos de NumPy, calculo estadísticos y hago transformaciones vectorizadas. La ventaja es que se hacen operaciones sobre toda una columna sin recorrerla fila por fila con un ciclo.

In [ ]:
# 1. Conversión de columnas a arreglos NumPy
scores = datos['imdb_score'].to_numpy(dtype=float)
votos = datos['imdb_votes'].to_numpy(dtype=float)

print('Tipo de scores:', type(scores))
print('Primeros cinco scores:', scores[:5])

# 2. Media y mediana
media_scores = np.mean(scores)
mediana_scores = np.median(scores)

# 3. Percentiles
p25, p75, p95 = np.percentile(scores, [25, 75, 95])

print('Media:', round(media_scores, 3))
print('Mediana:', round(mediana_scores, 3))
print('Percentil 25:', round(p25, 3))
print('Percentil 75:', round(p75, 3))
print('Percentil 95:', round(p95, 3))

In [ ]:
# 4. Operaciones vectorizadas sin ciclos
votos_log = np.log10(votos + 1)
calificacion_alta = scores >= 7.0
scores_estandarizados = (scores - np.mean(scores)) / np.std(scores)

print('Títulos con calificación de 7 o más:', int(np.sum(calificacion_alta)))
print('Porcentaje:', round(np.mean(calificacion_alta) * 100, 2), '%')
print('Primeros valores de votos en logaritmo:', np.round(votos_log[:5], 3))
print('Primeros scores estandarizados:', np.round(scores_estandarizados[:5], 3))

## 5. Análisis con SciPy

Se aplican dos procedimientos principales: correlación de Spearman y prueba de normalidad de Shapiro-Wilk. También se revisan posibles atípicos con z-score sobre los votos transformados con logaritmo.

In [ ]:
# Correlación de Spearman entre score y votos
rho, p_spearman = stats.spearmanr(datos['imdb_score'], datos['imdb_votes'])

print('Rho de Spearman:', round(rho, 4))
print('p-valor:', p_spearman)

if p_spearman < 0.05:
    print('Hay evidencia de una relación estadística entre las variables.')
else:
    print('No hay evidencia suficiente de una relación estadística.')

In [ ]:
# Prueba de normalidad Shapiro-Wilk con una muestra de máximo 5000 datos
muestra = datos['imdb_score'].sample(n=min(5000, len(datos)), random_state=42)
w_shapiro, p_shapiro = stats.shapiro(muestra)

print('Estadístico W:', round(w_shapiro, 4))
print('p-valor:', p_shapiro)

if p_shapiro < 0.05:
    print('La muestra no sigue una distribución normal.')
else:
    print('No se rechaza la normalidad de la muestra.')

In [ ]:
# Detección de posibles valores atípicos con z-score
datos['votos_log'] = np.log10(datos['imdb_votes'] + 1)
datos['z_votos'] = stats.zscore(datos['votos_log'])
atipicos = datos[datos['z_votos'].abs() > 3]

print('Posibles atípicos encontrados:', len(atipicos))
display(atipicos[['title', 'type', 'imdb_score', 'imdb_votes', 'z_votos']].head(10))

## 6. Visualización con Matplotlib

Se hacen tres gráficos diferentes: un histograma, un gráfico de dispersión y un boxplot.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(datos['imdb_score'], bins=25, color='steelblue', edgecolor='white')
plt.axvline(media_scores, color='red', linestyle='--', label='Media')
plt.axvline(mediana_scores, color='green', linestyle='--', label='Mediana')
plt.title('Distribución de calificaciones IMDb')
plt.xlabel('Calificación IMDb')
plt.ylabel('Cantidad de títulos')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(datos['imdb_score'], datos['imdb_votes'], alpha=0.3, s=15, color='darkorange')
plt.yscale('log')
plt.title('Calificación IMDb frente al número de votos')
plt.xlabel('Calificación IMDb')
plt.ylabel('Votos IMDb en escala logarítmica')
plt.show()

In [ ]:
tipos = sorted(datos['type'].dropna().unique())
grupos = [
    datos.loc[datos['type'] == tipo, 'imdb_score'].dropna()
    for tipo in tipos
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    grupos,
    tick_labels=tipos,
    patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
    medianprops=dict(color='darkred', linewidth=2)
)

plt.title('Calificación IMDb según el tipo de contenido')
plt.xlabel('Tipo')
plt.ylabel('Calificación IMDb')
plt.show()

## 7. Visualización con Seaborn

Ahora se repite el análisis visual con Seaborn. Esta librería permite crear gráficos estadísticos más rápido y con estilos ya organizados.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=datos, x='imdb_score', bins=25, kde=True, color='teal')
plt.title('Distribución de calificaciones con Seaborn')
plt.xlabel('Calificación IMDb')
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=datos, x='imdb_score', y='imdb_votes', hue='type', alpha=0.4)
plt.yscale('log')
plt.title('Calificación y votos según el tipo de contenido')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=datos, x='type', y='imdb_score')
plt.title('Comparación de calificaciones por tipo')
plt.xlabel('Tipo')
plt.ylabel('Calificación IMDb')
plt.show()

In [ ]:
columnas_corr = ['imdb_score', 'imdb_votes', 'tmdb_score', 'tmdb_popularity', 'runtime', 'release_year']
columnas_corr = [col for col in columnas_corr if col in datos.columns]

plt.figure(figsize=(8, 6))
sns.heatmap(datos[columnas_corr].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Mapa de correlaciones')
plt.show()

## 8. Uso complementario de credits.csv

El archivo de créditos permite revisar los roles y las personas relacionadas con los títulos. También se puede unir con la tabla principal mediante la columna `id`.

In [ ]:
print('Cantidad de registros por rol:')
display(credits['role'].value_counts())

print('Personas con más apariciones:')
display(credits['name'].value_counts().head(10))

In [ ]:
union = credits.merge(
    datos[['id', 'title', 'type', 'imdb_score']],
    on='id',
    how='inner'
)

display(union.head())
print('Registros después de unir las tablas:', len(union))

## 9. Pensamiento crítico

**¿Cuándo usar Matplotlib y cuándo Seaborn?** Matplotlib sirve cuando se necesita controlar casi todo en la gráfica, por ejemplo tamaños, colores o anotaciones específicas. Seaborn es útil cuando se busca hacer gráficos estadísticos de forma rápida. En este taller el mapa de calor fue más sencillo de hacer con Seaborn, mientras que el histograma de Matplotlib permitió agregar fácilmente las líneas de media y mediana.

**¿Qué aporta NumPy frente a trabajar solo con Pandas?** NumPy permite manejar arreglos numéricos y hacer cálculos vectorizados de forma rápida. Con él se pudieron calcular percentiles, transformar los votos con logaritmo y estandarizar los scores sin usar ciclos.

**¿Qué aportó SciPy?** SciPy aportó pruebas estadísticas que no son tan directas en Pandas o NumPy, como Shapiro-Wilk, el p-valor de Spearman y el cálculo de z-score para revisar posibles datos atípicos.

**¿Qué usaría para un modelo de IA?** Sumaria Scikit-learn porque tiene modelos, herramientas de preparación de datos y métricas de evaluación. Por ejemplo, se podría intentar estimar una categoría de calificación usando variables como año, duración, popularidad y votos.

**Limitaciones encontradas:** hay títulos sin score o sin votos, por lo que esos registros no se pueden usar en todos los análisis. Además, los votos tienen una distribución muy desigual: unos títulos tienen muy pocos y otros millones. Por eso se utilizó escala logarítmica en los gráficos y cálculos relacionados con votos.

## Conclusión

El taller permitió practicar el proceso básico de análisis de datos: cargar información, revisar su estructura, tratar algunos problemas de calidad, calcular estadísticas y visualizar resultados. También quedó claro que antes de intentar entrenar un modelo de IA es importante entender los datos, sus nulos, sus valores atípicos y las relaciones que pueden existir entre las variables.